# What is Congress actually working on?
**Topic clustering of bills in the 119th Congress (2025–2026) with TF-IDF + K-Means**

Plan:
1. Download official bill data (free, no API key) from GovInfo bulk data
2. Parse it into a table: title, sponsor, party, state, official policy area, CRS summary
3. Turn the summaries into TF-IDF vectors
4. Cluster them with K-Means and label each cluster by its top words
5. Check the clusters against Congress's own *policy area* labels (this is your validation for a paper)
6. Look up any member of Congress and see what topics their bills cover

Run the cells top to bottom. Works in Google Colab, Jupyter, or VS Code.

In [ ]:
# Colab already has these; on your own machine run this once
!pip install -q pandas scikit-learn requests lxml matplotlib

## 1. Download the data
GovInfo publishes one ZIP per bill type per Congress. `hr` = House bills, `s` = Senate bills.
Each ZIP holds one XML file per bill with its title, sponsor, cosponsors, actions, policy area and the Congressional Research Service (CRS) summary.

In [ ]:
import os, zipfile, requests

CONGRESS = 119              # 119th Congress = 2025-2026
BILL_TYPES = ["hr", "s"]    # House and Senate bills
DATA_DIR = "billstatus"
os.makedirs(DATA_DIR, exist_ok=True)

for bt in BILL_TYPES:
    url = f"https://www.govinfo.gov/bulkdata/BILLSTATUS/{CONGRESS}/{bt}/BILLSTATUS-{CONGRESS}-{bt}.zip"
    zpath = f"{DATA_DIR}/{CONGRESS}-{bt}.zip"
    if not os.path.exists(zpath):
        print("Downloading", url)
        r = requests.get(url, headers={"User-Agent": "bill-clustering-project", "Accept": "application/zip"}, timeout=300)
        r.raise_for_status()
        open(zpath, "wb").write(r.content)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(f"{DATA_DIR}/{bt}")
    print(bt, "->", len(os.listdir(f'{DATA_DIR}/{bt}')), "files")

## 2. Parse the XML into a table

In [ ]:
import glob, re, html
import xml.etree.ElementTree as ET
import pandas as pd

def clean(t):
    t = html.unescape(t or "")
    t = re.sub(r"<[^>]+>", " ", t)          # summaries contain HTML tags
    return re.sub(r"\s+", " ", t).strip()

def parse_bill(path):
    bill = ET.parse(path).getroot().find("bill")
    btype = bill.findtext("type") or bill.findtext("billType") or ""
    num = bill.findtext("number") or bill.findtext("billNumber") or ""
    sp = bill.find("sponsors/item")
    # use the most recent CRS summary if there are several versions
    sums = [s.findtext("text") for s in bill.findall("summaries/summary")] or \
           [s.findtext("text") for s in bill.findall("summaries/billSummaries/item")]
    return {
        "bill_id": f"{CONGRESS}-{btype.lower()}-{num}",
        "title": clean(bill.findtext("title")),
        "sponsor": sp.findtext("fullName") if sp is not None else None,
        "party": sp.findtext("party") if sp is not None else None,
        "state": sp.findtext("state") if sp is not None else None,
        "introduced": bill.findtext("introducedDate"),
        "policy_area": bill.findtext("policyArea/name"),
        "latest_action": clean(bill.findtext("latestAction/text")),
        "summary": clean(sums[-1]) if sums else "",
        "url": f"https://www.congress.gov/bill/{CONGRESS}th-congress/{'house' if btype.lower()=='hr' else 'senate'}-bill/{num}",
    }

files = glob.glob(f"{DATA_DIR}/*/*.xml")
df = pd.DataFrame([parse_bill(f) for f in files])
print(len(df), "bills total;", (df.summary != "").sum(), "have a CRS summary")
df.head()

## 3. Build the text and the TF-IDF matrix
Only bills with a CRS summary are used (newer bills may not have one yet). The text is title + summary.

Legal boilerplate words like *secretary*, *shall*, *requires* appear in almost every bill and would dominate the clusters, so they're added to the stop-word list. Look at your top terms later and add more if needed.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

data = df[df.summary.str.len() > 100].reset_index(drop=True)
data["text"] = data.title + ". " + data.summary

legal_words = {"bill", "act", "secretary", "shall", "section", "requires", "require", "required",
               "provides", "provide", "establishes", "including", "federal", "united", "states",
               "department", "program", "programs", "certain", "year", "fiscal", "authorizes",
               "specified", "must", "amends", "law", "purposes", "related", "respect", "addition"}
stop = list(ENGLISH_STOP_WORDS | legal_words)

vec = TfidfVectorizer(stop_words=stop, ngram_range=(1, 2), min_df=5, max_df=0.4,
                      sublinear_tf=True, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b")
X = vec.fit_transform(data.text)
terms = vec.get_feature_names_out()
print("Matrix:", X.shape, "(bills x terms)")

## 4. Pick the number of clusters
There's no single right *k*. The silhouette score (higher = better separated) gives a hint; also judge by whether the top words make sense to a human. Congress itself uses ~32 policy areas, so 15–35 is a reasonable range.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

ks, scores = list(range(10, 41, 5)), []
for k in ks:
    km = KMeans(n_clusters=k, n_init=5, random_state=42).fit(X)
    s = silhouette_score(X, km.labels_, metric="cosine", sample_size=min(3000, X.shape[0]), random_state=42)
    scores.append(s); print(k, round(s, 4))

plt.plot(ks, scores, marker="o"); plt.xlabel("k"); plt.ylabel("silhouette (cosine)"); plt.show()

In [ ]:
K = 25   # <- change after looking at the chart and the top words below
km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(X)
data["cluster"] = km.labels_

# top words per cluster = largest values in each cluster centre
top_words = {}
for c in range(K):
    idx = km.cluster_centers_[c].argsort()[::-1][:10]
    top_words[c] = ", ".join(terms[i] for i in idx)

summary = (data.groupby("cluster")
               .agg(bills=("bill_id", "count"),
                    most_common_policy_area=("policy_area", lambda s: s.mode().iat[0] if s.notna().any() else None))
               .assign(top_words=lambda d: d.index.map(top_words))
               .sort_values("bills", ascending=False))
pd.set_option("display.max_colwidth", 120)
summary

## 5. Name your clusters
Read the top words and a few example titles, then give each cluster a short human name. This step is where *you* add judgment, so do it carefully and write down how you decided.

In [ ]:
def show(c, n=5):
    print(f"Cluster {c}: {top_words[c]}")
    for t in data[data.cluster == c].title.sample(min(n, (data.cluster == c).sum()), random_state=1):
        print("   -", t[:110])

for c in summary.index[:5]:   # look at all of them eventually
    show(c); print()

In [ ]:
# Fill this in after reading the clusters, e.g. {0: "Veterans health", 3: "Tax credits", ...}
cluster_names = {}
data["topic"] = data.cluster.map(lambda c: cluster_names.get(c, f"Cluster {c}"))

## 6. Validate against Congress's own labels
Every bill gets one official *policy area* from CRS. If your clusters line up with those, the method works; where they *don't* line up is often the interesting finding (e.g. one policy area splitting into two very different kinds of bills). Adjusted Rand Index: 0 = random, 1 = identical.

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

v = data.dropna(subset=["policy_area"])
print("Adjusted Rand Index:", round(adjusted_rand_score(v.policy_area, v.cluster), 3))
print("Normalized Mutual Info:", round(normalized_mutual_info_score(v.policy_area, v.cluster), 3))

ct = pd.crosstab(v.topic, v.policy_area)
ct.loc[:, ct.sum().sort_values(ascending=False).index[:12]]   # 12 biggest policy areas

## 7. The useful part: look up a member of Congress
Type part of a name (as it appears in the data, e.g. `"Smith"`). This shows what topics that member's sponsored bills fall into.

In [ ]:
def member_profile(name):
    m = data[data.sponsor.fillna("").str.contains(name, case=False)]
    if m.empty:
        print("No bills found for", name); return
    print(m.sponsor.iloc[0], "-", len(m), "sponsored bills with summaries\n")
    print(m.topic.value_counts().to_string(), "\n")
    return m[["bill_id", "title", "topic", "latest_action", "url"]]

member_profile("Smith")

## 8. Save your results

In [ ]:
data.drop(columns=["text"]).to_csv(f"bills_{CONGRESS}_clustered.csv", index=False)
summary.to_csv(f"clusters_{CONGRESS}.csv")
print("Saved.")

## Next steps
- **Improve topics:** swap TF-IDF for sentence embeddings (`sentence-transformers`, model `all-MiniLM-L6-v2`) and compare scores in step 6. That comparison alone is a nice paper section.
- **More data:** add `hjres`, `sjres` bill types, or the 118th Congress to compare two Congresses.
- **Fresher data:** the free Congress.gov API (key from api.congress.gov/sign-up) has the latest bills before bulk files update.
- **Share it:** turn step 7 into a simple web page where anyone types their representative's name.
- **Stay neutral:** describe what bills are about; don't score or rank members. Always link the source bill.